# Gold validation (read-only)

Checks already-built Gold tables against Silver and the registry: row reconciliation (which also detects join
multiplication), grain uniqueness, key coverage, join-dimension uniqueness, date-key coverage in `dim_date`, and
table-specific checks. It writes nothing.

Result meanings: `PASS`/`FAIL` for rules the documentation or the design makes absolute; `OBSERVE` for counts that are
reported but are not errors (for example unmatched optional keys, reversed source timestamps that Silver already
flags); `SKIPPED` when a large scan was not confirmed.

Cost: tables flagged `scan=large` need `confirm_large_scan=true` (one aggregation pass plus one key-uniqueness pass
per table; labevents and chartevents are hundreds of millions of rows). A reviewed run is the evidence that
`validation_evidence` in `00_Gold_Registry` may be changed from `none`.

In [0]:
%run ./00_Gold_Registry

In [0]:
%run ./01_Gold_Engine

In [0]:
declare_widgets(dbutils)
cfg = read_config(dbutils)

In [0]:
apply_session(spark, cfg)
S, G = cfg["source_schema"], cfg["target_schema"]
plan = build_plan(cfg["requested"])
all_results = []
for step in plan["steps"]:
    print(f"\n=== {step['table']}")
    results = validate_table(spark, step["table"], S, G, cfg, confirm_large=cfg["confirm_large_scan"])
    print_results(results)
    all_results.extend(results)

counts = {}
for r in all_results:
    counts[r["status"]] = counts.get(r["status"], 0) + 1
print("\nSummary:", counts)
if failures(all_results):
    raise RuntimeError(f"Validation failures: {[(r['table'], r['check']) for r in failures(all_results)]}")

In [0]:
# Date coverage of dim_date itself (metadata-sized query).
if spark.catalog.tableExists(f"{G}.dim_date"):
    display(spark.sql(f"SELECT MIN(full_date) AS first_date, MAX(full_date) AS last_date, COUNT(*) AS days FROM {G}.dim_date"))